In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/AirQualityUCI.csv", sep=";", decimal=",")
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]  # drop trailing empty cols
df = df.dropna(how="all")                              # drop fully blank rows, if any
df["datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"], format="%d/%m/%Y %H.%M.%S"
)
df = df.drop(columns=["Date", "Time"]).set_index("datetime")
df = df.replace(-200, np.nan)

In [ ]:
df.head()

In [ ]:
s = df["CO(GT)"]
edges = [0, 1.5, 3, 6, 10, np.inf]

In [ ]:
print(df.isna().mean().sort_values(ascending=False).round(3))
isn = df["CO(GT)"].isna()
gap_len = isn.groupby((~isn).cumsum()).sum()
print(gap_len[gap_len > 0].describe())

In [ ]:
print(df[["PT08.S2(NMHC)", "C6H6(GT)"]].corr(method="spearman").round(3))
print(df[["PT08.S1(CO)", "CO(GT)"]].corr(method="spearman").round(3))

m = df[["PT08.S2(NMHC)", "C6H6(GT)"]].dropna()
plt.scatter(m.iloc[:, 0], m.iloc[:, 1], s=3, alpha=.25)
plt.xlabel("PT08.S2(NMHC)"); plt.ylabel("C6H6(GT)")
plt.savefig("fig_s2_vs_benzene.png", dpi=160, bbox_inches="tight")

In [ ]:
CO_EDGES = [0, 1.5, 3.0, np.inf]
y = pd.cut(df["CO(GT)"], CO_EDGES, right=False, labels=False)
print(y.value_counts().sort_index())

s = df["CO(GT)"].dropna()
plt.hist(s, bins=np.arange(0, 12.2, .2))
for e in CO_EDGES[1:-1]:
  plt.axvline(e, color="r", ls="--")
plt.xlabel("CO(GT) (mg/m^3)"); plt.ylabel("hours")
plt.savefig("fig_target_edges.png", dpi=160, bbox_inches="tight")

In [ ]:
features = ["PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)", "PT08.S4(NO2)",
            "PT08.S5(O3)", "T", "RH", "AH"]
edges = df[features].quantile([1/3, 2/3]).T.round(2)
print(edges)

fig, axs = plt.subplots(2, 4, figsize=(12, 5))
for ax, c in zip(axs.ravel(), features):
  ax.hist(df[c].dropna(), bins=50)
  for q in edges.loc[c]:
    ax.axvline(q, color="r", ls="--")
  ax.set_title(c)
plt.tight_layout(); plt.savefig("fig_features_terciles.png", dpi=160)

In [ ]:
d = pd.DataFrame({"y": y})
for c in ["PT08.S1(CO)", "T", "AH"]:
    d[c] = pd.qcut(df[c], 3, labels=False)

for parents in (["PT08.S1(CO)", "T"], ["PT08.S1(CO)", "T", "AH"]):
    g = (d.dropna(subset=["y"] + parents)
          .groupby(parents + ["y"], observed=False).size().unstack(fill_value=0))
    print(parents, "| configs:", len(g), "| min rows:", g.sum(axis=1).min(),
          "| configs <30 rows:", (g.sum(axis=1) < 30).sum(),
          "| empty cells %:", round((g == 0).mean().mean() * 100, 1))

**Target pollutant: CO(GT)**

I choose CO(GT) as the target because:


*   The 0.88 correlation between PTO8.S1(C) and CO(GT).
*   7,344 complete hours, and the 0.1-11.9 mg/m^3 range.
*   The daily pattern: 74% of 0-5h readings in the lowest level against 18% of 18-23h
*   At least 24% of hours in each of the 3 bins
*   Gap problem: 18% missing, with a 173-hour gap